In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/processed/nft_labeled_rules.csv")

rule_cols = [
    "rule_self_trade",
    "rule_seller_buyback",
    "rule_multi_hop_cycle",
    "rule_high_pair_count"
]

df[rule_cols].head()

C:\Users\VENTUS\AppData\Local\Temp\ipykernel_12136\3981035450.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/nft_labeled_rules.csv")


,rule_self_trade,rule_seller_buyback,rule_multi_hop_cycle,rule_high_pair_count
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,1
4,0,0,0,1


In [4]:
rule_summary = []

for col in rule_cols:
    triggered = df[col].sum()
    percentage = triggered / len(df) * 100
    
    rule_summary.append({
        "rule": col,
        "triggered_count": triggered,
        "percentage": percentage
    })

rule_summary_df = pd.DataFrame(rule_summary)
rule_summary_df

,rule,triggered_count,percentage
0,rule_self_trade,55,0.003688
1,rule_seller_buyback,1211,0.081195
2,rule_multi_hop_cycle,859,0.057594
3,rule_high_pair_count,339,0.022729


In [5]:
df["wash_score"] = df[rule_cols].sum(axis=1)

overlap_summary = (
    df["wash_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)

overlap_summary.columns = ["num_rules_triggered", "count"]
overlap_summary["percentage"] = overlap_summary["count"] / len(df) * 100

overlap_summary

,num_rules_triggered,count,percentage
0,0,1489978,99.900299
1,1,669,0.044855
2,2,659,0.044185
3,3,159,0.010661


In [6]:
def confidence_category(score):
    if score == 0:
        return "normal"
    elif score == 1:
        return "suspicious_low_confidence"
    elif score == 2:
        return "wash_medium_confidence"
    elif score == 3:
        return "wash_high_confidence"
    else:
        return "wash_very_high_confidence"

df["confidence_category"] = df["wash_score"].apply(confidence_category)

df["confidence_category"].value_counts()

confidence_category
normal                       1489978
suspicious_low_confidence        669
wash_medium_confidence           659
wash_high_confidence             159
Name: count, dtype: int64

In [7]:
df["label_final"] = np.nan

df.loc[df["wash_score"] == 0, "label_final"] = 0
df.loc[df["wash_score"] >= 2, "label_final"] = 1

df["label_final"].value_counts(dropna=False)

label_final
0.0    1489978
1.0        818
NaN        669
Name: count, dtype: int64

In [8]:
df_train_ready = df[df["label_final"].notna()].copy()

df_train_ready["label_final"] = df_train_ready["label_final"].astype(int)

df_train_ready["label_final"].value_counts()

label_final
0    1489978
1        818
Name: count, dtype: int64

In [9]:
normal_count = (df_train_ready["label_final"] == 0).sum()
wash_count = (df_train_ready["label_final"] == 1).sum()

ratio = normal_count / wash_count if wash_count > 0 else np.inf

print("Normal:", normal_count)
print("Wash:", wash_count)
print("Ratio normal/wash:", ratio)
print("Batas maksimal:", 50)

Normal: 1489978
Wash: 818
Ratio normal/wash: 1821.4889975550122
Batas maksimal: 50


In [10]:
combination_summary = (
    df[rule_cols]
    .astype(str)
    .agg("-".join, axis=1)
    .value_counts()
    .reset_index()
)

combination_summary.columns = ["rule_combination", "count"]
combination_summary["percentage"] = combination_summary["count"] / len(df) * 100

combination_summary

,rule_combination,count,percentage
0,0-0-0-0,1489978,99.900299
1,0-1-1-0,644,0.043179
2,0-1-0-0,393,0.026350
3,0-0-0-1,166,0.011130
4,0-1-1-1,159,0.010661
5,0-0-1-0,56,0.003755
6,1-0-0-0,54,0.003621
7,0-1-0-1,14,0.000939
8,1-1-0-0,1,0.000067


In [12]:
from pathlib import Path

Path("../outputs/metrics").mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv("../data/processed/nft_rule_analysis.csv", index=False)

df_train_ready.to_csv(
    "../data/processed/nft_confidence_filtered.csv",
    index=False
)

rule_summary_df.to_csv(
    "../outputs/metrics/rule_summary.csv",
    index=False
)

overlap_summary.to_csv(
    "../outputs/metrics/rule_overlap_summary.csv",
    index=False
)

combination_summary.to_csv(
    "../outputs/metrics/rule_combination_summary.csv",
    index=False
)

print("Saved all rule analysis files.")

Saved all rule analysis files.


In [ ]:
Setiap transaksi diberikan wash_score berdasarkan jumlah rule yang terpenuhi:

wash_score = jumlah rule yang aktif

Distribusi wash score:

Wash Score	    Jumlah Data	Interpretasi
0	            1.489.978	Normal
1	            669	        Suspicious low-confidence
2	            659	        Wash medium-confidence
3	            159	        Wash high-confidence

Hasil overlap analysis menunjukkan bahwa mayoritas transaksi mencurigakan memenuhi lebih dari satu rule. Hal ini mengindikasikan bahwa kombinasi rule lebih efektif dibanding penggunaan single-rule labeling.

Kombinasi rule yang paling dominan adalah:

Kombinasi Rule	                                Jumlah
Seller Buyback + Multi-hop Cycle	            644
Seller Buyback only	                            393
High Pair Count only	                        166
Seller Buyback + Multi-hop Cycle + Pair Count	159

Hasil tersebut menunjukkan bahwa Rule 1 dan Rule 3 memiliki overlap yang kuat dan saling mendukung dalam mengidentifikasi pola wash trading.

Berdasarkan hasil overlap analysis, digunakan confidence filtering sebagai berikut:

Wash Score	Label
0	Normal
1	Suspicious (tidak digunakan training)
≥2	Wash Trading

Setelah confidence filtering:

Label Final	                Jumlah Data
Normal	                    1.489.978
Wash Trading	            818
Suspicious (dikeluarkan)	669

Pendekatan ini digunakan untuk mengurangi noisy label dan meningkatkan reliability positive class pada proses training graph model.